# Miras Gezgini: internet fotoğraflarından 3D miras

[MegaScenes](https://megascenes.github.io) veri seti, Wikimedia Commons'taki serbest lisanslı fotoğraflardan binlerce yerin kamera konumlarını hesaplamış. Bu defter seçtiğin UNESCO mirasının fotoğraflarını indirir, 3D Gaussian Splatting ile eğitir ve siteye eklenecek zip'i verir.

**Çalışma zamanı > Çalışma zamanı türünü değiştir > GPU (T4 veya L4)** seçip hücreleri sırayla çalıştır. Sahne başına L4'te ~25 dk, T4'te ~50 dk sürer. Eğitim arka planda çalışır; biten sahneler 5. hücreyle indirilir.

Üretilen sahneler fotoğrafların lisansları gereği **CC BY-SA 4.0** ile paylaşılır; zip içindeki `credits.json` kullanılan Commons dosyalarını listeler.

In [ ]:
#@title 1. GPU kontrolü
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2. Kurulum (8–12 dk)
REPO_URL = "https://github.com/Oguzhan61-dilmac/360-vid2splat.git"  #@param {type:"string"}
import os
if not os.path.exists("/content/360-vid2splat"):
    !git clone -q {REPO_URL} /content/360-vid2splat
%cd /content/360-vid2splat
!git pull -q
!pip install -q numpy opencv-python-headless scipy tqdm ninja
!pip install -q pycolmap-cuda12==4.2.1
!pip install -q --no-build-isolation gsplat==1.5.3
!node -v || (apt-get -qq update && apt-get -qq install -y nodejs npm)

# gsplat CUDA çekirdeklerini şimdi derle (ilk seferde birkaç dakika sürer)
import torch
from gsplat import rasterization
n = 16
img, _, _ = rasterization(
    torch.randn(n, 3, device="cuda"), torch.randn(n, 4, device="cuda"), torch.rand(n, 3, device="cuda") * 0.1,
    torch.rand(n, device="cuda"), torch.rand(n, 3, device="cuda"),
    torch.tensor([[[1., 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 5], [0, 0, 0, 1]]], device="cuda"),
    torch.tensor([[[100., 0, 50], [0, 100, 50], [0, 0, 1]]], device="cuda"), 100, 100)
import pycolmap
print("Hazır. COLMAP GPU:", pycolmap.has_cuda, "| gsplat çıktı:", tuple(img.shape))

In [ ]:
#@title 3. Eğitimi arka planda başlat
#@markdown Virgülle ayır: ayasofya, efes, tac-mahal, sagrada-familia, pantheon, aziz-vasil, stonehenge, suleymaniye, karlov-koprusu, partenon, sfenks, westminster
SAHNELER = "ayasofya,efes,tac-mahal,sagrada-familia"  #@param {type:"string"}
ADIM = 20000  #@param {type:"integer"}
import subprocess
log = open('/content/miras_egitim.log', 'a')
proc = subprocess.Popen(f'python scripts/megascenes_batch.py --names "{SAHNELER}" --out /content/miras --steps {ADIM}',
                        shell=True, stdout=log, stderr=subprocess.STDOUT, cwd='/content/360-vid2splat')
print('Eğitim arka planda başladı. Durumu 4. hücreyle izle.')

In [ ]:
#@title 4. Durum
!grep -E "=====|seçilen|alt model|eğitim\] [0-9]+/|PSNR|hazır|HATA|Error|error" /content/miras_egitim.log | tail -n 12
!ls -la /content/miras/*.zip 2>/dev/null

In [ ]:
#@title 5. Biten zip'leri indir (daha önce inenleri atlar)
import glob, os
from google.colab import files
done = set(open('/content/indirilen.txt').read().split()) if os.path.exists('/content/indirilen.txt') else set()
for z in sorted(glob.glob('/content/miras/*.zip')):
    if z not in done:
        files.download(z)
        done.add(z)
open('/content/indirilen.txt', 'w').write('\n'.join(sorted(done)))
print(sorted(done))

In [ ]:
#@title 6. Önizlemeler (solda gerçek fotoğraf, sağda model)
import glob, json, zipfile
from IPython.display import Image, display
for z in sorted(glob.glob('/content/miras/*.zip')):
    with zipfile.ZipFile(z) as f:
        print(z, json.loads(f.read('metrics.json')))
        for n in [n for n in f.namelist() if n.startswith('previews/')][:2]:
            display(Image(f.read(n), width=900))